# Session 6: Clean Code & AI Refactoring

> Write self-documenting code and leverage CodeX/Claude for high-speed refactoring.

## 1. Meaningful Naming & Single Responsibility

**Why naming matters more than you think**
Code is read ~10× more often than it is written. A function named `proc(d, f)` forces every reader to mentally execute the body to understand what it does. A function named `get_eligible_users(users, active_only)` is self-documenting — the reader can understand the intent in one second without reading the body at all.

**Good naming rules:**
- Variables: use nouns that describe what the value *is* (`user_list`, not `data` or `lst`).
- Functions: use verb phrases that describe what the function *does* (`calculate_tax`, `validate_email`).
- Booleans: use `is_`, `has_`, `can_` prefixes (`is_active`, `has_permission`).
- Avoid abbreviations unless they are universally understood in your domain (`url`, `id`, `db`).

**Single Responsibility in functions**
A function should do one thing and do it completely. The test: can you name it without using "and"? `validate_and_save_user` does two things. Split it into `validate_user` and `save_user`. Each becomes shorter, easier to test, and easier to reuse.

In [ ]:
# ❌ Cryptic — requires mental decoding
def proc(d, f=False):
    r = []
    for i in d:
        if i['a'] > 18 and (not f or i['s'] == 'active'): r.append(i)
    return r

# ✅ Self-documenting — reads like a sentence
from typing import Optional
def get_eligible_users(
    users: list[dict],
    active_only: bool = False
) -> list[dict]:
    """Return users who are adults, optionally filtered to active accounts."""
    return [
        user for user in users
        if user['age'] > 18
        and (not active_only or user['status'] == 'active')
    ]

users = [{'age':25,'status':'active'},{'age':16,'status':'active'},{'age':30,'status':'inactive'}]
print('Active adults:', get_eligible_users(users, active_only=True))

## 2. No Magic Numbers, No Deep Nesting

**Magic numbers** are literal values (`750`, `12`, `1`) embedded directly in logic. They are dangerous because:
1. The reader has no idea what `750` means without reading surrounding context or documentation.
2. If the threshold changes, you must find and update every occurrence — `grep` misses none only if you're lucky.
3. They make tests brittle: `assert result == 750` tells you nothing about *why* 750 is the right value.

Fix: always give numbers names (`MIN_CREDIT_SCORE = 750`). Put them at the top of the file or in a `constants.py`.

**Deep nesting** (3+ levels of `if` inside `for` inside `if`) forces readers to mentally track multiple conditions simultaneously. The brain's working memory overflows at around 3 levels.

**Guard clauses** (early returns) are the standard fix. Instead of:
```python
if condition_a:
    if condition_b:
        if condition_c:
            # real logic here
```
Write:
```python
if not condition_a: return False
if not condition_b: return False
if not condition_c: return False
# real logic here — guaranteed all conditions met
```
The real logic is now at nesting level 0, and each guard documents one specific rejection reason.

In [ ]:
# ❌ Magic numbers + deep nesting
def check(u):
    if u['type'] == 1:
        if u['score'] > 750:
            if u['months'] >= 12:
                return True
    return False

# ✅ Named constants + early return (Guard Clauses)
MIN_CREDIT_SCORE = 750
MIN_ACCOUNT_MONTHS = 12
PREMIUM_USER_TYPE = 1

def is_eligible_for_premium_loan(user: dict) -> bool:
    if user['type'] != PREMIUM_USER_TYPE: return False
    if user['score'] < MIN_CREDIT_SCORE: return False
    if user['months'] < MIN_ACCOUNT_MONTHS: return False
    return True

user = {'type':1,'score':800,'months':18}
print('Loan eligible:', is_eligible_for_premium_loan(user))

## 💡 Additional Examples: Clean Code

**Example 1 — DRY: Order Total Calculator**
DRY (Don't Repeat Yourself) is not just about avoiding copy-paste. It's about having a *single source of truth* for every piece of business logic. When discount rates appear in three places (`if premium: total * 0.8`, `if member: total * 0.9`, ...), changing the premium discount from 20% to 25% requires finding and updating every occurrence. A `DISCOUNT_RATES` dict + helper functions means you change one line and the entire system is consistent.

**The cost of WET code:** every duplicated logic block is a future bug waiting to happen. Teams regularly spend hours debugging because one copy of duplicated logic was updated and another was not.

**Example 2 — Single-purpose functions: Registration pipeline**
Validation is the most commonly over-bundled responsibility. A `register_user` function that validates, creates, sends email, and logs — all in one body — means:
- You cannot test email validation without running the DB save logic.
- You cannot reuse the email validator for a password-reset flow.
- A bug in the email-sending step crashes a user who is already saved.

Splitting into `validate_email`, `validate_password_strength`, `validate_registration_input`, and `create_user_record` means each can be tested in 2 lines of test code.

**Example 3 — Comments: explain WHY, not WHAT**
`# loop through users` is noise — the `for u in users` already says that. The comment adds zero information and will become wrong the moment someone refactors the loop into a list comprehension.

Valuable comments explain things that are *invisible* from the code:
- Business rules: *why* a 30-day grace period exists (GDPR compliance).
- Non-obvious trade-offs: *why* you're using a slower algorithm (correctness under edge case X).
- External constraints: *why* a magic constant exists (third-party API rate limit).

In [ ]:
# ─── Example 1: DRY — Extract helper functions to avoid repeated logic ────────
from datetime import datetime, timezone
from typing import Optional

# ❌ WET (Write Everything Twice) — discount + tax logic duplicated in every branch
def calculate_order_total_bad(items: list, user_type: str) -> float:
    subtotal = sum(i['price'] * i['qty'] for i in items)
    if user_type == 'vip':
        discount = subtotal * 0.20
        tax = (subtotal - discount) * 0.10
        return subtotal - discount + tax
    elif user_type == 'member':
        discount = subtotal * 0.10
        tax = (subtotal - discount) * 0.10
        return subtotal - discount + tax
    else:
        discount = 0
        tax = subtotal * 0.10
        return subtotal - discount + tax

# ✅ DRY — named helpers, each doing one thing
DISCOUNT_RATES = {'vip': 0.20, 'member': 0.10, 'guest': 0.00}
TAX_RATE = 0.10

def calculate_subtotal(items: list[dict]) -> float:
    return sum(item['price'] * item['qty'] for item in items)

def apply_discount(amount: float, user_type: str) -> float:
    rate = DISCOUNT_RATES.get(user_type, 0.0)
    return amount * rate

def calculate_tax(amount_after_discount: float) -> float:
    return amount_after_discount * TAX_RATE

def calculate_order_total(items: list[dict], user_type: str) -> dict:
    subtotal = calculate_subtotal(items)
    discount = apply_discount(subtotal, user_type)
    taxable_amount = subtotal - discount
    tax = calculate_tax(taxable_amount)
    return {
        'subtotal': round(subtotal, 2),
        'discount': round(discount, 2),
        'tax': round(tax, 2),
        'total': round(taxable_amount + tax, 2),
    }

cart = [{'name': 'Laptop', 'price': 1200, 'qty': 1}, {'name': 'Mouse', 'price': 25, 'qty': 2}]
for utype in ['vip', 'member', 'guest']:
    result = calculate_order_total(cart, utype)
    print(f'{utype:8s}: subtotal={result["subtotal"]}, discount={result["discount"]}, total={result["total"]}')

# ─── Example 2: Single-purpose functions — Registration pipeline ──────────────
import re

# ❌ One function doing too many things at once
def register_user_bad(data):
    if not data.get('email') or '@' not in data['email']: return None
    if not data.get('password') or len(data['password']) < 8: return None
    if not data.get('name'): return None
    user = {'id': 1, **data, 'created_at': datetime.now(timezone.utc).isoformat()}
    print(f'DB: INSERT INTO users ... {user["email"]}')
    print(f'Email: Welcome {user["name"]}')
    return user

# ✅ Each function does exactly one thing — individually testable
def validate_email(email: str) -> bool:
    return bool(re.match(r'^[\w.+-]+@[\w-]+\.[a-z]{2,}$', email, re.IGNORECASE))

def validate_password_strength(password: str) -> Optional[str]:
    if len(password) < 8:              return 'Password must be at least 8 characters'
    if not re.search(r'\d', password): return 'Password must contain a digit'
    if not re.search(r'[A-Z]', password): return 'Password must contain an uppercase letter'
    return None  # None means valid

def validate_registration_input(data: dict) -> list[str]:
    errors = []
    if not validate_email(data.get('email', '')): errors.append('Invalid email address')
    pw_error = validate_password_strength(data.get('password', ''))
    if pw_error: errors.append(pw_error)
    if not data.get('name', '').strip(): errors.append('Name is required')
    return errors

def create_user_record(data: dict) -> dict:
    return {'id': 42, 'email': data['email'], 'name': data['name'],
            'created_at': datetime.now(timezone.utc).isoformat()}

test_data = {'email': 'alice@example.com', 'password': 'Secure123', 'name': 'Alice'}
errors = validate_registration_input(test_data)
if errors:
    print(f'Validation failed: {errors}')
else:
    user = create_user_record(test_data)
    print(f'✅ User created: {user["email"]} at {user["created_at"]}')

print('\nInvalid cases:')
print(validate_registration_input({'email': 'not-an-email', 'password': 'weak', 'name': ''}))

# ─── Example 3: Comments — explain WHY, not WHAT ─────────────────────────────
# ❌ Comments that restate what the code already says — pure noise
def get_active_users_bad(users):
    # Loop through users
    result = []
    # Check if user is active
    for u in users:
        if u['is_active']:  # If active
            result.append(u)  # Add to result
    return result  # Return result

# ✅ Comments that explain business decisions invisible from the code
INACTIVE_GRACE_PERIOD_DAYS = 30  # Compliance: GDPR requires a 30-day soft-delete window

def get_billable_users(users: list[dict]) -> list[dict]:
    """
    Return users eligible for billing in the current cycle.

    Note: Users inactive for < 30 days are still included — they remain
    on their paid plan during the grace period per the subscription contract (§4.2).
    Hard-deleted users (hard_deleted_at IS NOT NULL) are always excluded.
    """
    from datetime import timedelta
    grace_cutoff = datetime.now(timezone.utc) - timedelta(days=INACTIVE_GRACE_PERIOD_DAYS)
    return [
        u for u in users
        if not u.get('hard_deleted_at')  # Never bill hard-deleted accounts
        and (u['is_active'] or u.get('deactivated_at', datetime.now(timezone.utc)) > grace_cutoff)
    ]

sample_users = [
    {'name': 'Alice', 'is_active': True,  'hard_deleted_at': None},
    {'name': 'Bob',   'is_active': False, 'hard_deleted_at': None,
     'deactivated_at': datetime.now(timezone.utc)},   # Still within grace period
    {'name': 'Carol', 'is_active': False, 'hard_deleted_at': '2026-03-01'},  # Hard-deleted
]
billable = get_billable_users(sample_users)
print(f'\nBillable users: {[u["name"] for u in billable]}')  # Alice + Bob, not Carol


## 3. AI Lab: Refactoring Sprint

### 🧪 AI Lab / Practice

> **TODO:** Find a function >30 lines in your codebase → Use CodeX or Claude: 'Refactor this function applying: meaningful names, guard clauses instead of nesting, no magic numbers, single responsibility. Show before/after.' → Measure: lines reduced, cyclomatic complexity before/after.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')